In [ ]:
import importlib
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown

# -------------------- Dark Theme --------------------
display(HTML("""
<style>
    body, .jp-Notebook, .jp-OutputArea-output, .jp-RenderedHTMLCommon {
        background-color: #1e1e1e !important;
        color: #d4d4d4 !important;
    }
    h2, h3, h4 {
        color: #4fc3f7 !important;
        border-bottom: 2px solid #3498db !important;
        padding-bottom: 4px;
    }
    b, strong { color: #f48fb1 !important; }
    .highlight {
        background-color: #2d2d2d !important;
        padding: 10px;
        border-left: 4px solid #3498db;
        margin: 4px 0;
        color: #d4d4d4;
    }
    code {
        background-color: #333 !important;
        color: #ffcc80 !important;
        padding: 2px 4px;
        border-radius: 4px;
    }
    .dataframe {
        background-color: #2d2d2d !important;
        color: #d4d4d4 !important;
    }
</style>
"""))

# -------------------- Verbosity Flags --------------------
SHOW_VERBOSE = True
SHOW_INFO = True
SHOW_CRITICAL = True
SHOW_DEBUG = True

# -------------------- Helper functions for display --------------------
def display_title(title: str):
    display(HTML(f"<h2>{title}</h2>"))

def display_info(message: str):
    display(HTML(f"<div class='highlight'>{message}</div>"))

print("Environment ready. Dark theme applied.")

In [ ]:
# =============================================================================
# CELL 2 — CANONICAL DATASET DISCOVERY (drop-in, no registry module)
# =============================================================================

from pathlib import Path
import importlib
import Extraction as EX
importlib.reload(EX)

display_title("Processed dataset discovery")

DATASETS, CSV_HASHES = EX.discover_processed_datasets(
    datasets_root=EX.PROCESSED_DATASETS_ROOT,
    show_info=True,
)

display_info(
    f"<b>{len(DATASETS)}</b> processed datasets discovered under "
    f"<code>{EX.PROCESSED_DATASETS_ROOT}</code>"
)

# Visual summary
summary = pd.DataFrame([
    {
        "name": name,
        "rows": len(df),
        "columns": ", ".join(df.columns),
        "csv_sha256": CSV_HASHES[name][:16] + "…",
        "sample_label": str(df["label"].iloc[0]),
    }
    for name, df in DATASETS.items()
])
display(summary)

# Contract checks
for name, df in DATASETS.items():
    assert list(df.columns) == list(EX.PROCESSED_COLUMNS), f"{name}: bad columns"
    assert df.index.is_unique and df.index[0] == 0 and df.index[-1] == len(df) - 1, \
        f"{name}: index must be a clean RangeIndex so row i ↔ hidden_states[i]"
    assert df["label"].map(lambda x: isinstance(x, list) and len(x) > 0).all(), \
        f"{name}: all labels must be non-empty Python lists"

display_info("✅ All datasets satisfy the extraction contract.")

for name, df in DATASETS.items():
    display_title(f"Dataset: {name}")
    display_info(f"Shape: {df.shape}")
    display(df.head(2))
    # We'll rely on the probe's column detection later
    display_info(f"Samples: <b>{len(df):,}</b>")

display_title("Unified ID Scheme")
display_info("""
Each sample is assigned a unique integer ID (0..N-1) matching its row index.
This ID links:
• Input text (original DataFrame index)
• Hidden state vector (row in hidden_states.npy)
• Label (row in labels.npy)
No shuffling occurs, guaranteeing one‑to‑one mapping.
""")

In [ ]:
import importlib
import _shared
import Probe as probe

importlib.reload(_shared)
importlib.reload(probe)

# ──────────────────────────────────────────────────────────────────────────
# Contracts — resolved from _shared.contract_dict_for(dataset_name).
# One entry per dataset, always. Adding a new dataset requires zero edits
# to this cell; the resolver picks it up from the schema sidecar.
# ──────────────────────────────────────────────────────────────────────────
DATASET_CONTRACTS = {
    name: probe.DatasetContract(**_shared.contract_dict_for(name))
    for name in DATASETS
}

print("Resolved contracts:")
for name, c in DATASET_CONTRACTS.items():
    print(
        f"  {name:22s} target={c.target_type:12s} "
        f"task={c.task_type:12s} "
        f"classes={len(c.class_order) if c.class_order else 'derived'}"
    )

# ──────────────────────────────────────────────────────────────────────────
# Probe specs — the list that was missing.
#
# Four probes, in increasing capacity:
#   linear_logistic  — linear baseline; safest
#   mlp_1/2/3_hidden — nonlinear; each adds a hidden layer
#
# Keeping the same four across every (model, dataset) pair is what makes the
# layer curves comparable. Do not add a probe mid-matrix; the run_key depends
# on this list.
# ──────────────────────────────────────────────────────────────────────────
probes = [
    probe.ProbeSpec(
        name="linear_logistic",
        type="logistic",
        complexity="linear",
        standardize=True,
        C=1.0,
        max_iter=3000,
        selection_metric="macro_f1",
    ),
    probe.ProbeSpec(
        name="mlp_1_hidden",
        type="mlp",
        complexity="1_hidden",
        standardize=True,
        hidden_dims=["0.5d"],
        learning_rate=1e-3,
        weight_decay=1e-4,
        epochs=80,
        batch_size=256,
        patience=12,
        selection_metric="macro_f1",
    ),
    probe.ProbeSpec(
        name="mlp_2_hidden",
        type="mlp",
        complexity="2_hidden",
        standardize=True,
        hidden_dims=["0.5d", "0.25d"],
        learning_rate=1e-3,
        weight_decay=1e-4,
        epochs=80,
        batch_size=256,
        patience=12,
        selection_metric="macro_f1",
    ),
    probe.ProbeSpec(
        name="mlp_3_hidden",
        type="mlp",
        complexity="3_hidden",
        standardize=True,
        hidden_dims=["0.5d", "0.25d", "0.125d"],
        learning_rate=1e-3,
        weight_decay=1e-4,
        epochs=80,
        batch_size=256,
        patience=12,
        selection_metric="macro_f1",
    ),
]

print(f"\nProbes configured: {[p.name for p in probes]}")

In [ ]:
from pathlib import Path
import json
import pandas as pd
from _shared import HIDDEN_STATES_ROOT, INTEREX_ROOT, PROBE_ROOT


def discover_extraction_pairs(hidden_states_root: Path = HIDDEN_STATES_ROOT) -> pd.DataFrame:
    """Every (model, dataset) with a completed extraction under hidden_states/."""
    rows = []
    if not hidden_states_root.is_dir():
        return pd.DataFrame(columns=["model", "dataset", "artifact_dir"])

    for model_dir in sorted(hidden_states_root.iterdir()):
        if not model_dir.is_dir() or model_dir.name.startswith("_"):
            continue
        for dataset_dir in sorted(model_dir.iterdir()):
            if not dataset_dir.is_dir():
                continue
            meta_path = dataset_dir / "extraction.json"
            if not (dataset_dir / "hidden_states.npy").is_file() or not meta_path.is_file():
                continue
            try:
                meta = json.loads(meta_path.read_text())
            except Exception:
                continue
            rows.append({
                "model":        meta.get("model", {}).get("name") or model_dir.name,
                "dataset":      meta.get("dataset", {}).get("name") or dataset_dir.name,
                "artifact_dir": str(dataset_dir),
            })
    return pd.DataFrame(rows)


def discover_probe_runs(interex_root: Path = INTEREX_ROOT) -> pd.DataFrame:
    """Every probe run registered under interEx/<slug>/<dataset>/index.json."""
    rows = []
    if not interex_root.is_dir():
        return pd.DataFrame(columns=[
            "model", "dataset", "run_key", "trial_hash",
            "probes", "results_csv", "task_type", "n_classes",
        ])

    for model_dir in sorted(interex_root.iterdir()):
        if not model_dir.is_dir() or model_dir.name.startswith("_"):
            continue
        for dataset_dir in sorted(model_dir.iterdir()):
            if not dataset_dir.is_dir():
                continue
            index = dataset_dir / "index.json"
            if not index.is_file():
                continue
            try:
                payload = json.loads(index.read_text())
            except Exception:
                continue
            for entry in payload.get("runs", []):
                results_csv = dataset_dir / entry["results_csv"]
                rows.append({
                    "model":       entry["model"],
                    "dataset":     entry["dataset"],
                    "run_key":     entry["run_key"],
                    "trial_hash":  entry["trial_hash"],
                    "probes":      "+".join(entry["probes"]),
                    "results_csv": str(results_csv),
                    "task_type":   entry["task_type"],
                    "n_classes":   entry["n_classes"],
                })
    return pd.DataFrame(rows)


print(f"HIDDEN_STATES_ROOT = {HIDDEN_STATES_ROOT}")
print(f"PROBE_ROOT         = {PROBE_ROOT}")

In [ ]:
KNOWN_DATASETS = set(DATASETS.keys())
REPEATS        = 4
MAX_SAMPLES    = None
VERBOSE        = True
EXPERIMENT_ID  = "main_run"

pairs = discover_extraction_pairs()
print(f"Extraction artifacts found : {len(pairs)}")
if not pairs.empty:
    print(pairs.groupby("model").size().to_string())

entries = []
skipped_unknown = []
for row in pairs.itertuples(index=False):
    if row.dataset not in KNOWN_DATASETS:
        skipped_unknown.append((row.model, row.dataset))
        continue
    entries.append({
        "model":        row.model,
        "dataset":      row.dataset,
        "artifact_dir": row.artifact_dir,
        "contract":     DATASET_CONTRACTS[row.dataset],
        "dataset_df":   DATASETS[row.dataset],
    })

print(f"\nQueued for probing : {len(entries)}")
if skipped_unknown:
    print(f"Skipped {len(skipped_unknown)} pairs with no contract:")
    for m, d in skipped_unknown[:5]:
        print(f"  · {m} / {d}")

full_results = probe.run_matrix(
    entries,
    experiment_id=EXPERIMENT_ID,
    probes=probes,
    repeats=REPEATS,
    max_samples=MAX_SAMPLES,
    verbose=VERBOSE,
    checkpoint_dir=PROBE_ROOT / "_matrix_checkpoint",
    shuffled_label_control=True,
    shuffled_control_repeats=3,
)
print(f"\nMatrix completed. Full results shape: {full_results.shape}")

This custom, single_multi label should either go or imlemented, it's fine to use specific dataset and probe contracts for now, but the method needs to be regulated and one main schema to be produced. 

In [ ]:
runs = discover_probe_runs()
if runs.empty:
    print("No completed probe runs yet.")
    loaded = pd.DataFrame()
else:
    loaded = pd.concat(
        [pd.read_csv(p) for p in runs["results_csv"] if Path(p).is_file()],
        ignore_index=True,
    )
    print(f"Loaded {len(loaded):,} probe rows across {runs['run_key'].nunique()} runs.")
    display(runs[["model", "dataset", "probes", "task_type", "n_classes"]])

In [ ]:
# Prefer the in-memory matrix if cell 5 ran; else fall back to what's on disk.
df = full_results if not full_results.empty else loaded

if df.empty:
    print("Nothing to analyse yet. Run cell 5 first.")
else:
    best_per_probe = (
        df.loc[df.groupby(["probe", "model", "dataset"])["test_macro_f1"].idxmax()]
        .sort_values(["dataset", "model", "probe"])
    )
    display_title("Best Layer per Probe (Macro-F1)")
    display(best_per_probe[[
        "probe", "model", "dataset", "layer_index",
        "test_macro_f1", "probe_score",
    ]])

    pivot_best = best_per_probe.pivot_table(
        index=["model", "dataset"], columns="probe", values="test_macro_f1",
    )
    display_title("Best Macro-F1 Matrix")
    display(pivot_best.style.background_gradient(cmap="viridis", axis=None))

In [ ]:
output_plots_dir = Path("probe_plots")
output_plots_dir.mkdir(exist_ok=True)

if not full_results.empty:
    probe.create_final_visuals(full_results, output_plots_dir)
    print(f"Wrote per-metric plots to {output_plots_dir}/")
    for p in sorted(output_plots_dir.glob("*.png")):
        print(f"  {p.name}")
else:
    print("No results to plot.")

In [ ]:
df = full_results if not full_results.empty else loaded

if df.empty:
    print("Nothing to plot.")
else:
    out_dir = Path("probe_plots") / "per_pair"
    out_dir.mkdir(parents=True, exist_ok=True)

    for (model, dataset), group in df.groupby(["model", "dataset"]):
        fig, ax = plt.subplots(figsize=(11, 5))
        for probe_name in sorted(group["probe"].unique()):
            sub = group[group["probe"] == probe_name].sort_values("layer_index")
            ax.plot(sub["layer_index"], sub["test_macro_f1"],
                    marker="o", linewidth=2, label=probe_name)
        ax.set_xlabel("Layer index")
        ax.set_ylabel("Test Macro-F1")
        ax.set_title(f"{model} / {dataset} — layer-wise Macro-F1")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=9)
        fig.tight_layout()
        safe = f"{model.replace('/', '_')}__{dataset}.png"
        fig.savefig(out_dir / safe, dpi=200, bbox_inches="tight")
        plt.close(fig)
        print(f"  wrote {out_dir / safe}")